# Template do aluno: GridWorld 10×10 com SARSA, Q-Learning, Expected SARSA e TD(0)

**Aluno(a):** _________________________  **Versão do ambiente:** ____ (nome: _____________)

Este notebook segue **a mesma estrutura do notebook da aula** (`Experimentos_GridWorld_Custom_TD_MC_v2.ipynb`): mesmas seções, mesmas funções e mesmos nomes de variáveis. A diferença é que aqui **você completa as lacunas** e usa o **seu** ambiente. Use apenas os módulos do projeto: `environment.py`, `algorithms.py`, `visualization.py` e `compare_q_values.py` (na mesma pasta do notebook).

### Como usar
| Marca | Significado |
|---|---|
| ✏️ `# TODO` / `...` | lacuna que **você** completa (o `...` é só um marcador: substitua por código) |
| 💡 Dicas | ajuda para resolver a tarefa, sem entregar a resposta |
| 📋 O que colar | o que levar para o relatório (.docx) da sua versão |
| célula sem marcas | código **fornecido**: apenas execute (leia para entender) |

Execute **de cima para baixo**. Cada tarefa tem a mesma numeração do relatório (Tarefas 1 a 7), e as questões (discursiva e teste) estão no relatório, não aqui.

| Seção | Conteúdo | Tarefa do relatório |
|---|---|---|
| 1 | Teoria base | – |
| 2 | O ambiente (**o seu**) | 1 |
| 3 | Gabarito exato (Value Iteration) | 2 |
| 4 | Funções auxiliares de treino e exibição | – |
| 5–7 | SARSA, Q-Learning e Expected SARSA (γ = 0,9; noise = 0,2) | 3 |
| 8 | Comparação dos três algoritmos | 4 |
| 9 | TD(0): avaliação de uma política proposta | 5 |
| 10 | Sensibilidade a γ (Q-Learning) | 6 |
| 11 | Sensibilidade ao noise (Q-Learning) | 7 |
| 12 | Conclusões | – |

### Premissas (iguais para todas as versões)
1. **Terminais negativos (`x`): recompensa −10.** Prêmios: `A` = +20, `B` = +10, `C` = +5. Parede: `#`.
2. **Início de cada episódio:** célula livre não terminal sorteada (padrão dos módulos quando `initial_state=None`).
3. **Noise (estilo `environment.py`):** a ação pretendida ocorre com prob. `1 − noise` e cada perpendicular com `noise/2`. Bater em parede ou na borda mantém o agente no lugar.
4. **Hiperparâmetros fixos (não altere):** 5.000 episódios, **5 seeds** (0 a 4), α = 0,1, **ε = 0,1 constante**, custo de viver 0,05, γ = 0,9 e noise = 0,2 nas seções 5 a 9.
5. **Como ler as figuras do `visualize_gridworld`:** verde = terminal positivo, vermelho = terminal negativo, cinza = parede; o número na célula é V(s) e a **seta azul** é a ação da política.

## 1. Teoria base

**Aprendizado por Reforço (RL):** um agente aprende a decidir interagindo com um ambiente e recebendo recompensas. O objetivo é uma política que **maximize a recompensa cumulativa esperada**.

**MDP**, tupla $(S, A, P, R, \gamma)$: estados, ações, transição $P(s'|s,a)$, recompensa $R(s,a,s')$ e fator de desconto $\gamma \in [0,1]$.

**Retorno:**
$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

**Funções de valor** para uma política $\pi$:
$$V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s] \qquad Q^\pi(s,a) = \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a]$$

| Parâmetro | Papel | Efeito |
|---|---|---|
| **γ** (desconto) | pondera futuro × presente | γ→0: míope; γ→1: recompensas distantes quase tão importantes quanto as imediatas |
| **α** (taxa de aprendizado) | $\text{Novo} \leftarrow (1-\alpha)\,\text{Antigo} + \alpha\,\text{Estimativa}$ | α pequeno: lento e estável; α grande: rápido e instável |
| **ε** (exploração) | política ε-greedy: ação gulosa com prob. $1-\varepsilon+\varepsilon/\lvert A\rvert$, demais com $\varepsilon/\lvert A\rvert$ | ε=0: guloso; ε=1: aleatório. Aqui ε = 0.1 fixo |
| **noise** (do ambiente) | prob. de o movimento sair pelos lados | noise alto = ações menos confiáveis |

**Métodos TD** combinam Monte Carlo (aprendem da experiência, sem modelo) com Programação Dinâmica (*bootstrapping*: atualizam uma estimativa a partir de outra), e atualizam **a cada passo**.

## 2. O ambiente (Tarefa 1)

`create_custom_gridworld` recebe as paredes, os terminais, γ, noise e o `living_reward`. O custo de viver de 0,05 entra como `living_reward = -0.05`.

**Objetivo da tarefa:** reproduzir no notebook o gridworld da **sua versão** e conferir que ficou idêntico ao mapa do relatório.

> 💡 **Dicas — Tarefa 1**
>
> - Copie o **mapa em texto** da seção 1 do seu relatório para a lista `LAYOUT`: são 10 strings de 10 caracteres, uma por linha do grid (a linha 0 é a de cima).
> - Legenda do mapa: `#` parede, `x` armadilha (−10), `A` = +20, `B` = +10, `C` = +5, `.` célula livre. Confira que cada string tem **exatamente 10 caracteres**.
> - Em `build_gridworld`, as **paredes** são as posições `(r, c)` onde `LAYOUT[r][c] == '#'`, e os **terminais** formam um dicionário `{(r, c): recompensa}`: a recompensa vem de `RECOMPENSAS[LAYOUT[r][c]]`, para os caracteres que são chaves de `RECOMPENSAS`.
> - `create_custom_gridworld(ROWS, COLS, paredes, terminais, gamma=..., noise=..., living_reward=...)`: lembre que o custo de viver é **negativo**.
> - A célula de conferência compara o seu grid com os valores da tabela do relatório (paredes, terminais, total de estados). Se algum `assert` falhar, o mapa está diferente do enunciado: revise o `LAYOUT`.
> - `BOLSAO` são 5 células livres vizinhas de armadilhas (escolha no seu mapa olhando a figura). Elas só servem para imprimir valores de V* e das ações, nada mais depende delas.

> 📋 **O que colar no relatório**
>
> - o código de `LAYOUT` e `build_gridworld` (e de `BOLSAO`);
> - a figura gerada por `visualize_gridworld` (o seu grid);
> - a saída de `print_gridworld_info` (confira com a tabela do relatório).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Módulos do projeto
from environment import GridWorld, create_custom_gridworld, print_gridworld_info
from algorithms import td_zero_prediction, sarsa, q_learning, expected_sarsa, get_greedy_policy
from visualization import visualize_gridworld, visualize_q_values, plot_learning_curves, plot_value_heatmap, compare_algorithms
from compare_q_values import compare_q_values_by_states, find_most_divergent_states

# ---------- Parâmetros do enunciado (NÃO altere) ----------
ROWS, COLS   = 10, 10
LIVING_COST  = 0.05        # recompensa por passo = -0.05
NEG_TERMINAL = -10.0       # recompensa das armadilhas ('x')
GAMMA, NOISE = 0.9, 0.2    # valores de referência (seções 5 a 9)

# ---------- Hiperparâmetros de treino (NÃO altere) ----------
N_EPISODES, N_SEEDS = 5000, 5
ALPHA, EPSILON      = 0.1, 0.1     # ε constante: os módulos não têm decaimento
WINDOW              = 250          # janela da média móvel das curvas
FIGSIZE             = (10, 10)     # o padrão do visualize_gridworld seria 20x20 para um grid 10x10


# ✏️ TODO 1: copie aqui o mapa em texto da SUA versão (10 strings de 10 caracteres).
# '#' parede | 'x' armadilha (-10) | A=+20 | B=+10 | C=+5 | '.' célula livre
LAYOUT = [
    '??????????',   # linha 0
    '??????????',   # linha 1
    '??????????',   # linha 2
    '??????????',   # linha 3
    '??????????',   # linha 4
    '??????????',   # linha 5
    '??????????',   # linha 6
    '??????????',   # linha 7
    '??????????',   # linha 8
    '??????????',   # linha 9
]
RECOMPENSAS = {'A': 20.0, 'B': 10.0, 'C': 5.0, 'x': NEG_TERMINAL}


def build_gridworld(gamma=GAMMA, noise=NOISE):
    """Monta o grid 10x10 a partir do mapa LAYOUT. Coordenadas (linha, coluna); a linha 0 fica no topo."""
    # ✏️ TODO 2: paredes = posições (r, c) em que LAYOUT[r][c] é '#'
    walls = [(r, c) for r in range(ROWS) for c in range(COLS) if LAYOUT[r][c] == ...]
    # ✏️ TODO 3: terminais = {(r, c): recompensa} para os caracteres que estão em RECOMPENSAS
    terminals = {(r, c): ... for r in range(ROWS) for c in range(COLS) if LAYOUT[r][c] in ...}
    # ✏️ TODO 4: chame create_custom_gridworld (o custo de viver entra como NEGATIVO)
    return create_custom_gridworld(ROWS, COLS, ..., ..., gamma=gamma, noise=noise, living_reward=...)


gw_ref = build_gridworld()
visualize_gridworld(gw_ref, title="Meu ambiente (γ=0.9, noise=0.2, living cost 0.05)", figsize=FIGSIZE)
print_gridworld_info(gw_ref)


In [ ]:
# ✏️ TODO 5: conferência com a tabela da seção 1 do seu relatório
# (paredes, nº de terminais e total de estados = 100 − nº de paredes)
ESPERADO = {'paredes': ..., 'terminais': ..., 'estados': ...}

assert all(len(l) == COLS for l in LAYOUT) and len(LAYOUT) == ROWS, "LAYOUT deve ter 10 strings de 10 caracteres"
assert len(gw_ref.walls) == ESPERADO['paredes'], f"paredes: {len(gw_ref.walls)} no grid x {ESPERADO['paredes']} no relatório"
assert len(gw_ref.terminal_states) == ESPERADO['terminais'], f"terminais: {len(gw_ref.terminal_states)} no grid x {ESPERADO['terminais']} no relatório"
assert len(gw_ref.states) == ESPERADO['estados'], f"estados: {len(gw_ref.states)} no grid x {ESPERADO['estados']} no relatório"
livres_n = len([s for s in gw_ref.states if not gw_ref.is_terminal(s)])
print("Conferência OK. Células livres (nem parede, nem terminal):", livres_n)


**Leitura da topologia.** ✏️ Escreva aqui (3 a 5 linhas) como é o seu ambiente: onde ficam os prêmios (+20, +10, +5) e as armadilhas (−10), onde estão as paredes e quais são os caminhos mais curtos entre as regiões.

> 💡 Olhe a figura acima: o ambiente tem corredores estreitos? Há um prêmio “escondido” atrás de armadilhas ou paredes? Qual prêmio é o mais difícil de alcançar?

In [ ]:
# ✏️ TODO 6: escolha 5 células LIVRES vizinhas de armadilhas do seu mapa (só para imprimir V* e ações)
BOLSAO = [..., ..., ..., ..., ...]      # ex.: [(8, 1), (8, 2), ...]


## 3. Gabarito exato (Value Iteration) (Tarefa 2)

Os módulos não trazem uma solução exata. Como o ambiente é conhecido, `gw.get_all_transitions(s, a)` devolve a lista `(probabilidade, próximo_estado, recompensa)`, e com ela dá para calcular a solução ótima. O gabarito serve para **avaliar** o que os algoritmos aprenderam, e não para treiná-los.

- `value_iteration` aplica $V(s) \leftarrow \max_a \sum_{s'} P(s'|s,a)\,[R + \gamma V(s')]$ até convergir e devolve $V^*$ e $Q^*$.
- `policy_evaluation` calcula o $V^\pi$ **exato** de uma política fixa (a mesma recursão, sem o máximo). É a referência do TD(0).
- `avaliar_politica` mede, para a política gulosa de uma tabela Q: **% de ações ótimas** (uma ação conta como ótima se $Q^*(s,a)$ estiver a menos de 1e-6 do máximo, o que trata empates) e a **perda de valor** média $\overline{V^* - V^\pi}$.
- `mudancas_reais` (usada nas seções 10 e 11) lista as células em que **todas** as ações ótimas da configuração de referência perdem mais de 0.05 de valor na nova configuração. Isso separa mudança real de política de empate entre ações.

> 💡 **Dicas — Tarefa 2**
>
> - A célula seguinte é **fornecida**: só execute. Leia as funções para entender como o V* é calculado (equação de Bellman com máximo, repetida até convergir).
> - Ela imprime o V* nas 5 células de `BOLSAO` e o V* médio nos estados livres. Esse V* médio será usado no teste do relatório.
> - Na célula de consulta (logo abaixo), `V, Q = gabarito(gw_ref, GAMMA)` devolve `V[s]` (um número) e `Q[s]` (um dicionário `{ação: valor}`) para as células **livres**.
> - Escolha **uma célula vizinha de armadilha** e **uma célula próxima do +20**; compare V* e as ações ótimas de cada uma.

> 📋 **O que colar no relatório**
>
> - a saída da célula do gabarito (V* nas 5 células e V* médio);
> - o código e a saída da sua consulta a `V` e `Q` nas duas células escolhidas (diga quais são).

In [ ]:
_CACHE_T, _CACHE_G = {}, {}


def modelo_de_transicao(gw):
    """{(s, a): [(prob, s', recompensa), ...]} via gw.get_all_transitions (depende do layout e do noise, não de γ)."""
    chave = gw.noise
    if chave not in _CACHE_T:
        _CACHE_T[chave] = {(s, a): gw.get_all_transitions(s, a)
                           for s in gw.states if not gw.is_terminal(s) for a in gw.actions}
    return _CACHE_T[chave]


def valor_acao(T, V, s, a, gamma):
    return sum(p * (r + gamma * V[s2]) for p, s2, r in T[(s, a)])


def value_iteration(gw, gamma, tol=1e-10):
    """Retorna (V*, Q*) exatos. V* é um dict {estado: valor}; Q* é {estado: {ação: valor}}."""
    T = modelo_de_transicao(gw)
    livres = [s for s in gw.states if not gw.is_terminal(s)]
    V = {s: 0.0 for s in gw.states}                        # V(terminal) = 0
    while True:
        delta = 0.0
        for s in livres:
            novo = max(valor_acao(T, V, s, a, gamma) for a in gw.actions)
            delta = max(delta, abs(novo - V[s]))
            V[s] = novo
        if delta < tol:
            break
    Q = {s: {a: valor_acao(T, V, s, a, gamma) for a in gw.actions} for s in livres}
    return V, Q


def gabarito(gw, gamma):
    """value_iteration com cache por (noise, γ)."""
    chave = (gw.noise, gamma)
    if chave not in _CACHE_G:
        _CACHE_G[chave] = value_iteration(gw, gamma)
    return _CACHE_G[chave]


def policy_evaluation(gw, policy, gamma, tol=1e-10):
    """V^π exato de uma política determinística {estado: ação}."""
    T = modelo_de_transicao(gw)
    V = {s: 0.0 for s in gw.states}
    while True:
        delta = 0.0
        for s in policy:
            novo = valor_acao(T, V, s, policy[s], gamma)
            delta = max(delta, abs(novo - V[s]))
            V[s] = novo
        if delta < tol:
            break
    return V


def avaliar_politica(gw, Q, gamma, tol=1e-6):
    """Compara a política gulosa de Q com o gabarito: % de ações ótimas e perda média de valor."""
    V_star, Q_star = gabarito(gw, gamma)
    pol = get_greedy_policy(Q, gw)
    otimas = [Q_star[s][pol[s]] >= max(Q_star[s].values()) - tol for s in pol]
    V_pi = policy_evaluation(gw, pol, gamma)
    perda = np.mean([V_star[s] - V_pi[s] for s in pol])
    return {'pct_otimas': 100 * np.mean(otimas), 'perda_valor': perda}


def mudancas_reais(cfg, ref, tol=0.05):
    """Células onde TODAS as ações ótimas da referência perdem mais de tol de valor na configuração cfg."""
    _, Q_cfg = gabarito(cfg['gw'], cfg['gamma'])
    _, Q_ref = gabarito(ref['gw'], ref['gamma'])
    saida = []
    for s in Q_cfg:
        otimas_ref = [a for a, q in Q_ref[s].items() if q >= max(Q_ref[s].values()) - 1e-6]
        perda = max(Q_cfg[s].values()) - max(Q_cfg[s][a] for a in otimas_ref)
        if perda > tol:
            saida.append((s, round(perda, 2)))
    return saida


V_star_ref, Q_star_ref = gabarito(gw_ref, GAMMA)
print("V* em 5 células vizinhas de armadilhas:", {s: round(V_star_ref[s], 3) for s in BOLSAO}, "| valor de nunca terminar:", -LIVING_COST / (1 - GAMMA))
print("V* médio nos estados livres:", round(np.mean([V_star_ref[s] for s in Q_star_ref]), 3))

In [ ]:
# ✏️ TODO 7: consulte o gabarito em duas células do seu ambiente
V, Q = gabarito(gw_ref, GAMMA)
celula_armadilha = ...      # uma célula livre vizinha de armadilha, ex.: (8, 3)
celula_premio    = ...      # uma célula livre próxima do prêmio de +20

for s in [celula_armadilha, celula_premio]:
    melhor = max(Q[s], key=Q[s].get)
    print(f"{s}: V* = {...:.3f} | Q* = { {a: round(q, 3) for a, q in ....items()} } | melhor ação = {melhor}")


## 4. Funções auxiliares de treino e exibição

Cada algoritmo (`sarsa`, `q_learning`, `expected_sarsa`) devolve `(Q, recompensas_por_episódio)`. Elas **não têm parâmetro de seed**, e o módulo usa o gerador global do NumPy. Por isso `treinar` chama `np.random.seed(seed)` antes de cada execução, o que torna os resultados reprodutíveis.

- `treinar` roda **5 seeds** e guarda a **média das tabelas Q** (dela sai a política) e a **média das curvas de recompensa** (é o que vai para `plot_learning_curves`, que desenha uma curva por rótulo).
- O **gamma é passado explicitamente** a cada algoritmo, porque as funções de controle usam o argumento `gamma` e **não** `gw.gamma`.
- `mostrar` desenha as curvas de treino e o gridworld com V(s) = max_a Q(s,a) e a política gulosa, e imprime as métricas contra o gabarito.

In [ ]:
def treinar(algoritmo, gw, gamma, n_seeds=N_SEEDS):
    """Treina n_seeds execuções independentes. Retorna a média das Q, a média das curvas e o contexto."""
    Qs, curvas = [], []
    for seed in range(n_seeds):
        np.random.seed(seed)                                   # seed global usada pelos módulos
        Q, recompensas = algoritmo(gw, n_episodes=N_EPISODES, alpha=ALPHA, gamma=gamma, epsilon=EPSILON)
        Qs.append(Q)
        curvas.append(recompensas)
    return {'Q': np.mean(Qs, axis=0), 'recompensas': np.mean(curvas, axis=0), 'curvas': np.array(curvas), 'gw': gw, 'gamma': gamma}


def valores_de_Q(Q, gw):
    """V(s) = max_a Q(s,a) como dict {estado: valor}, o formato que visualize_gridworld espera."""
    return {s: float(np.max(Q[s[0] * gw.cols + s[1]])) for s in gw.states if not gw.is_terminal(s)}


def mostrar(res, nome):
    """Curvas de treino + gridworld com valores e política + métricas contra o gabarito."""
    gw = res['gw']
    plot_learning_curves({nome: res['recompensas']}, window=WINDOW,
                         title=f"{nome}: curvas de treinamento (média de {N_SEEDS} seeds)")
    V, pi = valores_de_Q(res['Q'], gw), get_greedy_policy(res['Q'], gw)
    visualize_gridworld(gw, values=V, policy=pi, figsize=FIGSIZE,
                        title=f"{nome}: V(s) e política ótima aprendida (γ={res['gamma']}, noise={gw.noise})")
    m = avaliar_politica(gw, res['Q'], res['gamma'])
    print(f"{nome}: {m['pct_otimas']:.0f}% das ações são ótimas | perda média de valor (V* − Vπ) = {m['perda_valor']:.3f} | "
          f"retorno médio nos últimos 500 episódios = {np.mean(res['recompensas'][-500:]):.2f}")

### Teste curto (antes dos treinos completos)

Os módulos não limitam os passos por episódio. Se alguma política aprendida ficar presa (por exemplo, batendo numa parede para sempre), o episódio só acaba por **exploração (ε) ou escorregão**, e pode ficar longo. Esta célula treina cada algoritmo por 500 episódios e **cronometra**. O pior episódio dá uma ideia do custo: cada passo custa −0.05, e a recompensa do terminal entra no total.

In [ ]:
import time
for nome, alg in [('SARSA', sarsa), ('Q-Learning', q_learning), ('Expected SARSA', expected_sarsa)]:
    np.random.seed(0)
    t0 = time.time()
    _, r = alg(gw_ref, n_episodes=500, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON)
    dt = time.time() - t0
    print(f"{nome:>15}: 500 episódios em {dt:4.1f}s | pior episódio: {min(r):7.1f} "
          f"| estimativa para 5000 ep × {N_SEEDS} seeds: {dt * 10 * N_SEEDS / 60:.1f} min")

# Tarefa 3: treino de SARSA, Q-Learning e Expected SARSA (seções 5, 6 e 7)

**Objetivo:** treinar os três algoritmos no seu ambiente (γ = 0,9; noise = 0,2; 5.000 episódios; 5 seeds) e mostrar, para cada um, as curvas de treinamento, o gridworld com V(s) e a política aprendida.

> 💡 **Dicas — Tarefa 3**
>
> - Cada algoritmo tem uma célula de **teoria** (já escrita) e uma célula de **treino**. Antes de cada treino, escreva **com suas palavras** o comentário teórico pedido (3 a 5 linhas): o que o algoritmo aprende, como é o alvo da atualização, se é on-policy ou off-policy.
> - `treinar(algoritmo, gw, gamma)` recebe **a função** do algoritmo (`sarsa`, `q_learning` ou `expected_sarsa`, sem parênteses), o grid e o γ, e devolve um dicionário com `Q`, `recompensas`, etc.
> - Guarde cada resultado em `RES` com o **nome do algoritmo** como chave; a seção 8 usa `RES.items()` e depende desses nomes: `'SARSA'`, `'Q-Learning'`, `'Expected SARSA'`.
> - `mostrar(resultado, nome)` desenha as curvas, o grid com V(s) e a política e imprime as métricas (% de ações ótimas, perda de valor, retorno).
> - Ao comparar as figuras de SARSA e Q-Learning, procure células **próximas das armadilhas** em que as setas diferem. Anote as coordenadas: você vai precisar delas na questão discursiva.

> 📋 **O que colar no relatório**
>
> - os três comentários teóricos;
> - o código das três células de treino;
> - as figuras de curvas e as figuras do grid (uma de cada por algoritmo);
> - as três linhas de métricas impressas por `mostrar`.

## 5. SARSA (on-policy)

✏️ **Comentário teórico (escreva com suas palavras, 3 a 5 linhas):** o que o SARSA aprende? Como é o alvo da atualização? Por que ele é on-policy?

*(escreva aqui)*

> 💡 Compare com a fórmula do SARSA: $Q(S_t,A_t) \leftarrow Q(S_t,A_t)+\alpha[R_{t+1}+\gamma Q(S_{t+1},A_{t+1})-Q(S_t,A_t)]$. Note qual ação aparece no alvo.

In [ ]:
RES = {}
# ✏️ TODO 8: treine o SARSA (função sarsa) no gw_ref e guarde em RES['SARSA']
RES['SARSA'] = treinar(..., ..., ...)
mostrar(RES['SARSA'], 'SARSA')


## 6. Q-Learning (off-policy)

✏️ **Comentário teórico (escreva com suas palavras, 3 a 5 linhas):** o que o Q-Learning aprende? O que muda no alvo em relação ao SARSA? Por que é off-policy?

*(escreva aqui)*

> 💡 O alvo usa $\max_a Q(S_{t+1},a)$. Pense: a ação executada no próximo passo importa para o alvo?

In [ ]:
# ✏️ TODO 9: faça o mesmo para o Q-Learning
RES[...] = treinar(..., ..., ...)
mostrar(RES[...], ...)


## 7. Expected SARSA

✏️ **Comentário teórico (escreva com suas palavras, 3 a 5 linhas):** o que o Expected SARSA usa no alvo? Em que ele se parece com o SARSA e com o Q-Learning?

*(escreva aqui)*

> 💡 O alvo usa a **esperança** de $Q$ no próximo estado sob a política ε-greedy: uma média ponderada pelas probabilidades de cada ação.

In [ ]:
# ✏️ TODO 10: faça o mesmo para o Expected SARSA
...
...
...


## 8. Comparação dos três algoritmos (Tarefa 4)

| Característica | SARSA | Q-Learning | Expected SARSA |
|---|---|---|---|
| Tipo | on-policy | off-policy | ambos |
| Alvo da atualização | $Q(S',A')$ | $\max_a Q(S',a)$ | $\mathbb{E}_\pi[Q(S',a)]$ |
| Comportamento | cauteloso | otimista/agressivo | balanceado |
| Exploração afeta os valores | sim | não | ponderada |
| Custo por atualização | $O(1)$ | $O(\lvert A\rvert)$ | $O(\lvert A\rvert)$ |

Abaixo: curvas sobrepostas, a tabela de métricas contra o gabarito e as funções de `compare_q_values.py`, que apontam **os estados em que os algoritmos mais divergem**.

> 💡 **Dicas — Tarefa 4**
>
> - Na primeira célula, `plot_learning_curves` recebe um dicionário `{nome: curva_de_recompensas}`; a tabela usa `avaliar_politica(gw, Q, gamma)`, que devolve `pct_otimas` e `perda_valor`.
> - `find_most_divergent_states(Q_dict, gw_ref, n_states=5)` devolve os estados em que os algoritmos mais discordam no valor V(s). Use esses estados em `compare_q_values_by_states`.
> - Na célula de diagnóstico, a política trivial “sempre N” vem de uma tabela Q de **zeros** com uma linha por estado do grid (`rows * cols`) e 4 colunas (uma por ação).
> - O retorno médio **por seed** usa `r['curvas']`, uma matriz `(seeds × episódios)`; pegue os **últimos 500 episódios** (`[:, -500:]`) e tire a média ao longo dos episódios (`axis=1`).
> - Lembre: a % de ações ótimas só faz sentido comparada com a linha de base “sempre N”, e o retorno de treino mistura qualidade da política com a sorte do estado inicial.

> 📋 **O que colar no relatório**
>
> - a figura das curvas sobrepostas e a tabela de métricas;
> - a saída de `find_most_divergent_states` e o gráfico das células mais divergentes;
> - a saída do diagnóstico (política “sempre N” e tabela de retorno por seed).

In [ ]:
# ✏️ TODO 11: complete a tabela de métricas (uma linha por algoritmo)
plot_learning_curves({n: r['recompensas'] for n, r in RES.items()}, window=WINDOW,
                     title=f"Comparação: SARSA × Q-Learning × Expected SARSA (γ={GAMMA}, noise={NOISE})")

linhas = {}
for n, r in RES.items():
    m = avaliar_politica(..., ..., ...)                      # grid, tabela Q e γ de cada resultado
    linhas[n] = {'% ações ótimas': ...,                      # chaves de m: 'pct_otimas' e 'perda_valor'
                 'perda de valor (V*−Vπ)': ...,
                 'retorno médio (últimos 500 ep.)': np.mean(...)}   # últimos 500 valores de r['recompensas']
pd.DataFrame(linhas).T.round(3)


In [ ]:
# ✏️ TODO 12: células em que os algoritmos mais divergem
Q_dict = {n: r['Q'] for n, r in RES.items()}
divergentes = find_most_divergent_states(..., ..., n_states=5)      # (estado, divergência max−min de V)
print("Estados com maior divergência de V(s) entre os algoritmos:")
for s, d in divergentes:
    print(f"  {s}: Δ = {d:.3f}")

df_div = compare_q_values_by_states(Q_dict, gw_ref, selected_states=[s for s, _ in divergentes],
                                    show_actions=False, show_differences=True, plot_comparison=True)


In [ ]:
compare_algorithms(Q_dict, gw_ref)    # tabela completa de V(s) = max_a Q(s,a) por estado e estatísticas resumidas

### Diagnóstico: o quanto dá para confiar nas diferenças

Duas verificações antes de comparar os algoritmos:

1. **Linha de base:** uma política trivial "sempre N" (o desempate do `argmax` quando Q é zero). Se ela já acertasse quase tudo, a % de ações ótimas dos algoritmos não diria nada.
2. **Dispersão entre seeds:** o retorno médio dos últimos 500 episódios, seed por seed. Com só 5 seeds, uma diferença entre algoritmos só merece atenção se for bem maior que essa dispersão.

Lembrete: o retorno por episódio mistura qualidade da política com **sorte do estado inicial** (sorteado entre as células livres), por isso é uma medida ruidosa.

In [ ]:
# (i) Linha de base: Q = 0 em todo lugar => argmax cai na ação 0 ('N') em todos os estados
# ✏️ TODO 13: tabela Q de zeros com uma linha por estado do grid (rows*cols) e 4 colunas (ações)
base = avaliar_politica(gw_ref, np.zeros((..., ...)), GAMMA)
print(f"Política trivial 'sempre N': {base['pct_otimas']:.0f}% de ações ótimas | perda de valor = {base['perda_valor']:.3f}")

# (ii) Retorno médio dos últimos 500 episódios, por seed
# ✏️ TODO 14: r['curvas'] é (seeds × episódios): últimos 500 episódios, média ao longo dos episódios
por_seed = {n: r['curvas'][..., ...].mean(axis=...) for n, r in RES.items()}
tab = pd.DataFrame({n: {**{f'seed {k}': v for k, v in enumerate(vals)}, 'média': vals.mean(), 'desvio entre seeds': vals.std()}
                    for n, vals in por_seed.items()})
tab.round(2)


## 9. TD(0): avaliação de uma política proposta (Tarefa 5)

**Predição** (não controle): dada uma política fixa $\pi$, estimar $V^\pi$ com

$$V(S_t) \leftarrow V(S_t) + \alpha\big[R_{t+1} + \gamma V(S_{t+1}) - V(S_t)\big], \qquad \delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$$

O termo entre colchetes é o **erro TD** $\delta_t$. Diferente do Monte Carlo (que espera o retorno real $G_t$ no fim do episódio), o TD(0) atualiza **a cada passo** usando a estimativa de $V(S_{t+1})$ (*bootstrapping*): menor variância, com algum viés. Vale ainda $G_t - V(S_t) = \sum_k \gamma^k \delta_{t+k}$.

```
Para cada episódio:
  Inicializar S
  Para cada passo:
    A ← π(S); executar A, observar R, S'
    V(S) ← V(S) + α [ R + γ V(S') − V(S) ]
    S ← S'
  até S ser terminal                 # V(terminal) = 0
```

### Política proposta

> 💡 **Dicas — Tarefa 5**
>
> - Você pode usar a política gulosa do Q-Learning (`get_greedy_policy(Q, gw)`, como na aula) ou escrever uma política à mão: um dicionário `{estado: ação}` para as células livres, com ações `'N'`, `'S'`, `'L'`, `'O'`.
> - A política **precisa levar todo episódio a um terminal**. `td_zero_prediction` não limita os passos: uma política que nunca termina (por exemplo, empurrar contra uma parede para sempre) deixa o programa em **laço infinito**. Por isso a célula `episodios_que_nao_terminam` vem **antes** do TD(0). Se a saída não for “nenhum”, corrija a política e anote o que mudou.
> - `td_zero_prediction(gw, policy, n_episodes=..., alpha=...)` devolve um dicionário `V`. Rode com as **5 seeds** (use `np.random.seed(seed)` antes de cada rodada) e tire a **média** entre elas para cada estado.
> - O V^π **exato** vem de `policy_evaluation(gw, policy, gamma)`: é a referência. O erro por célula é `abs(V_td[s] - V_pi[s])`; o RMSE de uma seed é `sqrt(mean((V[s] - V_pi[s])**2))` sobre as células livres.
> - Na convergência, o RMSE é medido repetindo o TD(0) com números crescentes de episódios; observe onde a curva **para de cair**.

> 📋 **O que colar no relatório**
>
> - a política proposta (código e figura) e a saída de `episodios_que_nao_terminam`;
> - as figuras V^π exato, V estimado por TD(0) e o mapa de erro;
> - o RMSE de cada seed e os V médios impressos;
> - a figura e a tabela de convergência.

In [ ]:
def episodios_que_nao_terminam(gw, policy, n_ep=50, max_passos=2000):
    """{estado_inicial: nº de episódios (de n_ep) que não chegaram a um terminal em max_passos passos}."""
    ruins = {}
    for s0 in policy:
        n = 0
        for _ in range(n_ep):
            s = s0
            for _ in range(max_passos):
                s, _r = gw.sample_transition(s, policy[s])
                if gw.is_terminal(s):
                    break
            else:
                n += 1
        if n:
            ruins[s0] = n
    return ruins




In [ ]:
# ✏️ TODO 15: defina a política proposta {estado: ação} para as células livres
pi_td = ...        # ex.: get_greedy_policy(RES['Q-Learning']['Q'], gw_ref)

np.random.seed(123)
print("Ações em células vizinhas de armadilhas (política proposta):", {s: pi_td[s] for s in BOLSAO})
print("Episódios que não terminam em 2.000 passos:", episodios_que_nao_terminam(gw_ref, pi_td) or "nenhum")

visualize_gridworld(gw_ref, policy=pi_td, figsize=FIGSIZE, title="Política proposta para o TD(0)")


### TD(0) com 5 seeds e comparação com o V^π exato

`td_zero_prediction` usa `gw.gamma` (0.9) e devolve um dict `V`. Uso **α = 0.05 e 20.000 episódios**: com α constante, o TD(0) fica oscilando em torno do valor verdadeiro, e um α menor reduz esse ruído. Cada seed gera um `V`, e a figura mostra a média. O **V^π exato** (`policy_evaluation`) é a referência.

In [ ]:
# ✏️ TODO 16: TD(0) com 5 seeds e comparação com o V^π exato
ALPHA_TD, N_TD = ..., ...             # enunciado: α = 0.05 e 20.000 episódios

V_runs = []
for seed in range(N_SEEDS):
    np.random.seed(seed)
    V_runs.append(td_zero_prediction(gw_ref, ..., n_episodes=..., alpha=...))
livres = list(pi_td)
V_td = {s: float(np.mean([... for V in V_runs])) for s in livres}      # média das seeds
V_pi = policy_evaluation(..., ..., ...)                                 # V^π exato (gabarito do TD(0))
erro = {s: abs(... - ...) for s in livres}

visualize_gridworld(gw_ref, values=V_pi, policy=pi_td, figsize=FIGSIZE, title="V^π exato (policy evaluation)")
visualize_gridworld(gw_ref, values=V_td, policy=pi_td, figsize=FIGSIZE, title=f"V^π estimado por TD(0) (média de {N_SEEDS} seeds)")
visualize_gridworld(gw_ref, values=erro, figsize=FIGSIZE, title="|V_TD(0) − V^π| por estado")

# RMSE de cada seed: sqrt(média de (V[s] − V_pi[s])² nas células livres)
rmse_por_seed = [np.sqrt(np.mean([... for s in livres])) for V in V_runs]
print("RMSE de cada seed:", np.round(rmse_por_seed, 3), "| erro máximo da média das seeds:", round(max(erro.values()), 3))
print(f"V médio nos estados livres: π proposta (exato) = {np.mean([V_pi[s] for s in livres]):.3f} | "
      f"TD(0) = {np.mean([V_td[s] for s in livres]):.3f} | ótimo V* = {np.mean([V_star_ref[s] for s in livres]):.3f}")


In [ ]:
plot_value_heatmap(V_td, gw_ref, title="TD(0): V(s) estimado (heatmap)", figsize=(11, 10))

**Convergência.** `td_zero_prediction` só devolve o `V` final, então a convergência é medida repetindo a execução com números crescentes de episódios, e não por uma curva dentro de um treino. O gráfico mostra o RMSE contra o V^π exato (média ± desvio entre as 5 seeds).

In [ ]:
# ✏️ TODO 17: convergência do TD(0): RMSE contra o número de episódios
tamanhos = [100, 300, 1000, 3000, 10000, 30000]
rmse = np.zeros((N_SEEDS, len(tamanhos)))
for j, n in enumerate(tamanhos):
    for seed in range(N_SEEDS):
        np.random.seed(seed)
        V = td_zero_prediction(gw_ref, ..., n_episodes=..., alpha=ALPHA_TD)
        rmse[seed, j] = np.sqrt(np.mean([... for s in livres]))     # mesmo RMSE da célula anterior

plt.figure(figsize=(7, 4))
plt.errorbar(tamanhos, rmse.mean(0), yerr=rmse.std(0), marker='o', capsize=3)
plt.xscale('log'); plt.yscale('log')
plt.xlabel('Número de episódios'); plt.ylabel('RMSE vs V^π exato')
plt.title(f'TD(0): convergência (α={ALPHA_TD}, média ± desvio, {N_SEEDS} seeds)')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(pd.DataFrame({'RMSE médio': rmse.mean(0), 'desvio': rmse.std(0)}, index=tamanhos).round(3))


## 10. Sensibilidade a γ (Q-Learning) (Tarefa 6)

γ pondera quanto o agente "enxerga" o futuro. Uma recompensa +20 a $k$ passos de distância vale cerca de $20\,\gamma^k$ **hoje**, então com γ menor o terminal distante de +20 perde atratividade frente a terminais mais próximos (+10 e +5).

**Como é feito:** para cada γ cria-se um **grid novo** (`gamma=g`) e passa-se o **mesmo γ ao `q_learning`**, porque as funções de controle ignoram `gw.gamma`. O caso γ = 0.9 reaproveita o treino da seção 6, que usa a mesma configuração e as mesmas seeds. Noise fixo em 0.2.

**Hipótese:** com γ menor, algumas células passam a preferir terminais próximos de valor menor, e V(s) cai em todo o grid. Abaixo de cada figura de política, `mudancas_reais` lista as células em que a ação ótima exata da referência (γ = 0.9) perde mais de 0.05 de valor na nova configuração.

> 💡 **Dicas — Tarefa 6**
>
> - Para cada γ, crie um grid novo com `build_gridworld(gamma=g, noise=NOISE)` e passe o **mesmo γ** a `treinar` (os algoritmos de controle ignoram `gw.gamma`). O caso γ = 0,9 já foi treinado na seção 6: reaproveite `RES['Q-Learning']`.
> - `valores_de_Q(r['Q'], r['gw'])` dá o V(s) para a figura, e `get_greedy_policy(r['Q'], r['gw'])` dá as setas.
> - `mudancas_reais(r, RES_G[0.9])` lista as células em que a ação ótima da referência perde valor com o novo γ. Se aparecerem células, descreva a **região** (por exemplo, “as células do lado esquerdo passam a ir para o +5”).
> - Pense em $20\,\gamma^k$: um prêmio a muitos passos vale muito menos com γ pequeno.

> 📋 **O que colar no relatório**
>
> - a figura das curvas para os três valores de γ;
> - as três figuras de política;
> - as linhas de “mudanças reais” impressas e a tabela de sensibilidade.

In [ ]:
GAMMAS = [0.7, 0.8, 0.9]
RES_G = {}
for g in GAMMAS:
    # ✏️ TODO 18: se g == GAMMA reaproveite RES['Q-Learning']; senão treine o Q-Learning num grid novo com esse γ
    RES_G[g] = RES['Q-Learning'] if g == GAMMA else treinar(..., build_gridworld(gamma=..., noise=NOISE), ...)

plot_learning_curves({f'γ = {g}': RES_G[g]['recompensas'] for g in GAMMAS}, window=WINDOW,
                     title=f"Q-Learning: sensibilidade a γ (noise = {NOISE})")


In [ ]:
for g in GAMMAS:
    r = RES_G[g]
    # ✏️ TODO 19: V(s) e política gulosa de cada resultado
    visualize_gridworld(r['gw'], values=..., policy=...,
                        figsize=FIGSIZE, title=f"Q-Learning: V(s) e política ótima com γ = {g} (noise = {NOISE})")
    print(f"γ = {g}: mudanças reais vs. γ = 0.9 -> {mudancas_reais(r, RES_G[...]) or 'nenhuma'}")


In [ ]:
def tabela_sensibilidade(resultados, rotulo, ref):
    linhas = {}
    for k, r in resultados.items():
        m = avaliar_politica(r['gw'], r['Q'], r['gamma'])
        V_star, Q_star = gabarito(r['gw'], r['gamma'])
        linhas[k] = {'% ações ótimas': m['pct_otimas'], 'perda de valor': m['perda_valor'],
                     'retorno médio (últ. 500)': np.mean(r['recompensas'][-500:]),
                     'mudanças reais vs. ref.': len(mudancas_reais(r, resultados[ref])),
                     'V* médio (exato)': np.mean([V_star[s] for s in Q_star])}
    return pd.DataFrame(linhas).T.rename_axis(rotulo).round(3)


tabela_sensibilidade(RES_G, 'γ', ref=0.9)

In [ ]:
# ✏️ TODO 20: tabela de sensibilidade a γ (referência: γ = 0.9)
tabela_sensibilidade(..., 'γ', ref=...)


## 11. Sensibilidade ao noise (Q-Learning) (Tarefa 7)

O noise controla a **confiabilidade das ações**. Com noise alto, andar rente às armadilhas de −10 fica perigoso: um deslize lateral pode ser fatal. Percorrer corredores longos até o +20 também fica mais difícil.

**Como é feito:** para cada noise cria-se um **grid novo** (`noise=n`), em vez de alterar `gw.noise` do grid original, para o valor não vazar para outras células. γ fixo em 0.9, e o caso noise = 0.2 reaproveita o treino da seção 6.

**Hipótese:** com mais noise, as políticas se afastam das armadilhas, e o aprendizado inicial fica mais lento.

> As curvas de retorno de noises diferentes **não medem qualidade** de forma direta, porque o ambiente fica mais difícil quando o noise sobe (o retorno ótimo cai). O que se compara é a forma da curva e a política obtida.

> 💡 **Dicas — Tarefa 7**
>
> - É o mesmo esquema da Tarefa 6, agora variando o **noise**: para cada `n` crie `build_gridworld(gamma=GAMMA, noise=n)` e treine o Q-Learning com `GAMMA`. O caso `n == NOISE` (0,2) reaproveita `RES['Q-Learning']`.
> - Compare as figuras de política de noise 0,2 e 0,6: as setas tendem a **se afastar das armadilhas** e a preferir rotas mais seguras (mesmo que mais longas).
> - `mudancas_reais(r, RES_N[0.2])` mostra as células em que a política mudou de verdade. Cite as mais afetadas e explique pelo risco de **escorregar** para uma armadilha (probabilidade `noise/2` para cada lado).
> - Não compare qualidade pelas curvas de retorno: com noise alto, o retorno **ótimo** também cai (o ambiente fica mais difícil).

> 📋 **O que colar no relatório**
>
> - a figura das curvas para os cinco valores de noise;
> - as cinco figuras de política;
> - as linhas de “mudanças reais” impressas e a tabela de sensibilidade.

In [ ]:
NOISES = [0.2, 0.3, 0.4, 0.5, 0.6]
RES_N = {}
for n in NOISES:
    # ✏️ TODO 21: se n == NOISE reaproveite RES['Q-Learning']; senão treine num grid novo com esse noise (γ = GAMMA)
    RES_N[n] = RES['Q-Learning'] if n == NOISE else treinar(..., build_gridworld(gamma=..., noise=...), ...)

plot_learning_curves({f'noise = {n}': RES_N[n]['recompensas'] for n in NOISES}, window=WINDOW,
                     title=f"Q-Learning: sensibilidade ao noise (γ = {GAMMA})")


In [ ]:
for n in NOISES:
    r = RES_N[n]
    # ✏️ TODO 22: V(s) e política gulosa de cada resultado; mudanças reais em relação a noise = 0.2
    visualize_gridworld(r['gw'], values=..., policy=...,
                        figsize=FIGSIZE, title=f"Q-Learning: V(s) e política ótima com noise = {n} (γ = {GAMMA})")
    print(f"noise = {n}: mudanças reais vs. noise = 0.2 -> {mudancas_reais(..., ...) or 'nenhuma'}")


In [ ]:
# ✏️ TODO 23: tabela de sensibilidade ao noise (referência: noise = 0.2)
tabela_sensibilidade(..., 'noise', ref=...)


## 12. Conclusões

✏️ Escreva aqui as suas conclusões (uma frase ou duas por item), **citando números das suas saídas**:

1. **Algoritmos:** qual teve mais ações ótimas e menor perda de valor no seu ambiente? A diferença foi grande em relação à dispersão entre seeds e à linha de base “sempre N”?
2. **TD(0):** em quantos episódios o erro parou de cair? Onde ficou o maior erro?
3. **γ:** o que mudou na política quando γ diminuiu? Em que região?
4. **Noise:** o que mudou quando o noise subiu? Que células mudaram mais e por quê?
5. **Limitações:** o que você não conseguiu concluir com 5 seeds e ε constante?

*(escreva aqui)*

> Entrega: este notebook **executado de ponta a ponta, sem erros**, junto com o relatório (.docx) da sua versão.